In [26]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats
from scipy.stats import entropy
import datetime


In [27]:
WINDOW_SIZE = 1.5

In [28]:
x_axis = 'x_ema'
y_axis = 'y_ema'
z_axis = 'z_ema'

In [29]:
NUMBER_OF_THRESHOLDS = 50
MIN_THRESHOLD = 0.00001
MAX_THRESHOLD = 1
t = np.linspace(MIN_THRESHOLD, 1, NUMBER_OF_THRESHOLDS)

threshold_list_linear = t * MAX_THRESHOLD
threshold_list_linear = threshold_list_linear.round(4)
threshold_list_linear

threshold_list = (t ** 3) * MAX_THRESHOLD
threshold_list = threshold_list.round(4)
threshold_list

array([0.000e+00, 0.000e+00, 1.000e-04, 2.000e-04, 5.000e-04, 1.100e-03,
       1.800e-03, 2.900e-03, 4.400e-03, 6.200e-03, 8.500e-03, 1.130e-02,
       1.470e-02, 1.870e-02, 2.330e-02, 2.870e-02, 3.480e-02, 4.180e-02,
       4.960e-02, 5.830e-02, 6.800e-02, 7.870e-02, 9.050e-02, 1.034e-01,
       1.175e-01, 1.328e-01, 1.494e-01, 1.673e-01, 1.866e-01, 2.073e-01,
       2.295e-01, 2.532e-01, 2.785e-01, 3.055e-01, 3.341e-01, 3.644e-01,
       3.966e-01, 4.305e-01, 4.664e-01, 5.042e-01, 5.440e-01, 5.858e-01,
       6.297e-01, 6.758e-01, 7.241e-01, 7.746e-01, 8.273e-01, 8.825e-01,
       9.400e-01, 1.000e+00])

In [30]:
def recurrence_plot_vector(data, eps):
    N = len(data)
    recurrence_matrix = np.zeros((N, N))

    # Calculate the Euclidean distance between vectors
    for i in range(N):
        for j in range(i, N):
            distance = np.linalg.norm(data[i] - data[j])
            recurrence_matrix[i, j] = recurrence_matrix[j, i] = distance

    # Apply the threshold
    recurrence_matrix = (recurrence_matrix <= eps).astype(int)
    
    return recurrence_matrix

In [31]:
def plot_recurrence_by_activity(data, dataset_name, axis_name, threshold, step, first=100, padding=2.0, filename=None):
    num_keys = len(data)
    num_plots_per_key = 6
    fig, axes = plt.subplots(num_keys, num_plots_per_key, figsize=(20, 20))
    
    # Extracting information from filename for the figure title
    figure_title = f'{dataset_name} - {axis_name} - {str(threshold)}'
    fig.suptitle(figure_title, fontsize=20, fontweight="bold")  # Set the figure title

    for i, (key, value) in enumerate(data.items()):
        start = first
        for j in range(num_plots_per_key):
            activity_data = value[start:start + step]
            R = recurrence_plot_vector(activity_data, threshold)
            ax = axes[i, j]
            cax = ax.matshow(R, cmap='binary')
            ax.set_title(f'{key} - Plot {j + 1}')
            ax.set_xticks([])
            ax.set_yticks([])
            start = start + int(step / 2)

    plt.tight_layout(pad=padding, rect=[0, 0, 1, 0.96])  # Adjust padding and fit the title

    if filename:
        plt.savefig(filename + ".png")  # Save the figure
        plt.close(fig)  # Close the figure to prevent it from displaying
    else:
        plt.show()  # Show the figure

# WISDM

In [32]:
DATASET_NAME = 'WISDM'
RELATIVE_PATH = "C:/Users/tulio/Downloads/Plots/WISDM"
WISDM_HZ = 20
WISDM_START = int(WINDOW_SIZE * WISDM_HZ * 3)

In [33]:
# Function to normalize columns
def normalize_group(group):
    group[['x', 'y', 'z']] = (group[['x', 'y', 'z']] - group[['x', 'y', 'z']].mean()) / group[['x', 'y', 'z']].std()
    return group

wisdm = pd.read_csv('../../datasets/WISDM_ar_v1.1_balanced.csv', header=None, sep=',', names=['user', 'activity', 'timestamp', 'x', 'y', 'z'])
wisdm = wisdm.groupby('user').apply(normalize_group)
wisdm = wisdm.reset_index(drop=True)
wisdm

C:\Users\tulio\AppData\Local\Temp\ipykernel_2484\471678201.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  wisdm = wisdm.groupby('user').apply(normalize_group)


,user,activity,timestamp,x,y,z
0,1,Walking,4992322339000,-0.091720,1.294345,-0.181997
1,1,Walking,4992422345000,0.390024,0.596599,-0.299797
2,1,Walking,4992722333000,1.040662,0.803481,0.175850
3,1,Walking,4993792341000,0.976051,0.650568,-0.224227
4,1,Walking,4993842298000,1.449861,0.680123,0.549256
...,...,...,...,...,...,...
290359,36,Standing,15049012250000,0.430103,0.422333,-0.155289
290360,36,Standing,15049062268000,0.330533,0.436234,-0.166769
290361,36,Standing,15049112287000,0.212525,0.439709,-0.155289
290362,36,Standing,15049162275000,0.002322,0.307651,-0.254785


In [34]:
wisdm[['x_maf', 'y_maf', 'z_maf']] = wisdm.groupby('user')[['x', 'y', 'z']].rolling(window=WISDM_HZ).mean().reset_index(drop=True)
wisdm[['x_ema', 'y_ema', 'z_ema']] = wisdm.groupby('user')[['x', 'y', 'z']].ewm(span=1.5*WISDM_HZ, adjust=False).mean().reset_index(drop=True)

In [35]:
walking_data = wisdm[(wisdm['activity'] == 'Walking') & (wisdm['user'] == 3)]
walking_vectors = walking_data[[x_axis, y_axis, z_axis]].to_numpy()

standing_data = wisdm[(wisdm['activity'] == 'Standing') & (wisdm['user'] == 3)]
standing_vectors = standing_data[[x_axis, y_axis, z_axis]].to_numpy()

sitting_data = wisdm[(wisdm['activity'] == 'Sitting') & (wisdm['user'] == 3)]
sitting_vectors = sitting_data[[x_axis, y_axis, z_axis]].to_numpy()

jogging_data = wisdm[(wisdm['activity'] == 'Jogging') & (wisdm['user'] == 3)]
jogging_vectors = jogging_data[[x_axis, y_axis, z_axis]].to_numpy()

upstairs_data = wisdm[(wisdm['activity'] == 'Upstairs') & (wisdm['user'] == 3)]
upstairs_vectors = upstairs_data[[x_axis, y_axis, z_axis]].to_numpy()

downstairs_data = wisdm[(wisdm['activity'] == 'Downstairs') & (wisdm['user'] == 3)]
downstairs_vectors = downstairs_data[[x_axis, y_axis, z_axis]].to_numpy()


print(len(walking_vectors)), print(len(standing_vectors)), print(len(sitting_vectors)), print(len(jogging_vectors)), print(len(upstairs_vectors)), print(len(downstairs_vectors))

1488
2824
1312
1607
1334
1534


(None, None, None, None, None, None)

In [36]:
data_x = {"Standing": standing_data[x_axis].to_numpy(), 
          "Walking": walking_data[x_axis].to_numpy(), 
          "Sitting": sitting_data[x_axis].to_numpy(), 
          "Jogging": jogging_data[x_axis].to_numpy(),
          "Downstairs": downstairs_data[x_axis].to_numpy(),
          "Upstairs": upstairs_data[x_axis].to_numpy()
          }

data_y = {"Standing": standing_data[y_axis].to_numpy(), 
          "Walking": walking_data[y_axis].to_numpy(), 
          "Sitting": sitting_data[y_axis].to_numpy(), 
          "Jogging": jogging_data[y_axis].to_numpy(),
            "Downstairs": downstairs_data[y_axis].to_numpy(),
            "Upstairs": upstairs_data[y_axis].to_numpy()
          }

data_z = {"Standing": standing_data[z_axis].to_numpy(), 
          "Walking": walking_data[z_axis].to_numpy(), 
          "Sitting": sitting_data[z_axis].to_numpy(), 
          "Jogging": jogging_data[z_axis].to_numpy(),
            "Downstairs": downstairs_data[z_axis].to_numpy(),
            "Upstairs": upstairs_data[z_axis].to_numpy()
          }

data_vectors = {"Standing": standing_vectors, 
                "Walking": walking_vectors, 
                "Sitting": sitting_vectors, 
                "Jogging": jogging_vectors,
                "Downstairs": downstairs_vectors,
                "Upstairs": upstairs_vectors
                }

In [37]:
STEP = int(WINDOW_SIZE * WISDM_HZ)
for i in range(len(threshold_list)):
    plot_recurrence_by_activity(data_x, DATASET_NAME, "X", threshold_list[i], step=STEP, first=WISDM_START, filename=f"{RELATIVE_PATH}/{WINDOW_SIZE}/X/recurrence_wisdm_x_{i}-{str(threshold_list[i])}")
    plot_recurrence_by_activity(data_y, DATASET_NAME, "Y", threshold_list[i], step=STEP, first=WISDM_START, filename=f"{RELATIVE_PATH}/{WINDOW_SIZE}/Y/recurrence_wisdm_y_{i}-{str(threshold_list[i])}")
    plot_recurrence_by_activity(data_z, DATASET_NAME, "Z", threshold_list[i], step=STEP, first=WISDM_START, filename=f"{RELATIVE_PATH}/{WINDOW_SIZE}/Z/recurrence_wisdm_z_{i}-{str(threshold_list[i])}")
    plot_recurrence_by_activity(data_vectors, DATASET_NAME, "Vector", threshold_list[i], step=STEP, first=WISDM_START, filename=f"{RELATIVE_PATH}/{WINDOW_SIZE}/Vector/recurrence_wisdm_vector_{i}-{str(threshold_list[i])}")

# PAMAP2

In [38]:
DATASET_NAME = 'PAMAP2'
RELATIVE_PATH = "C:/Users/tulio/Downloads/Plots/PAMAP2"
PAMAP2_HZ = 100
PAMAP2_START = int(WINDOW_SIZE * PAMAP2_HZ * 3)

In [39]:
# Function to normalize columns
def normalize_group(group):
    group[['hand_acc_16g_x', 'hand_acc_16g_y', 'hand_acc_16g_z']] = (group[['hand_acc_16g_x', 'hand_acc_16g_y', 'hand_acc_16g_z']] - group[['hand_acc_16g_x', 'hand_acc_16g_y', 'hand_acc_16g_z']].mean()) / group[['hand_acc_16g_x', 'hand_acc_16g_y', 'hand_acc_16g_z']].std()
    return group

pamap2 = pd.read_csv('../../datasets/pamap_balanced.csv')
pamap2 = pamap2.groupby('subject_id').apply(normalize_group)
pamap2 = pamap2.reset_index(drop=True)
pamap2

C:\Users\tulio\AppData\Local\Temp\ipykernel_2484\3837000985.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pamap2 = pamap2.groupby('subject_id').apply(normalize_group)


,timestamp,activity_id,hand_acc_16g_x,hand_acc_16g_y,hand_acc_16g_z,hand_acc_6g_x,hand_acc_6g_y,hand_acc_6g_z,hand_gyro_x,hand_gyro_y,hand_gyro_z,hand_mag_x,hand_mag_y,hand_mag_z,subject_id
0,309.53,2,1.173981,0.391292,0.007209,2.424750,9.12390,3.02166,0.037944,0.024816,-0.016013,15.0033,-67.00450,-1.055520,1
1,309.54,2,1.205812,0.395961,-0.151766,2.394240,9.13917,2.96125,0.040065,0.011347,0.009483,14.5708,-66.98840,-0.356026,1
2,309.55,2,1.189279,0.390786,-0.120353,2.378790,9.21468,2.84029,0.017566,-0.017100,-0.006770,14.6740,-67.31640,-0.707928,1
3,309.56,2,1.183728,0.374697,-0.104401,2.377660,9.13912,2.78006,-0.047912,0.007181,0.005693,14.7736,-67.42350,-1.288870,1
4,309.57,2,1.213794,0.391098,0.040929,2.361820,9.04862,2.78027,-0.046287,-0.000137,0.012163,15.1053,-66.56780,-1.630630,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
926023,3630.07,5,1.268499,0.884639,2.541252,0.374732,2.89221,9.30422,-0.017195,0.006944,0.007758,17.7034,-4.55562,-41.179600,8
926024,3630.08,5,1.250323,0.870380,2.488375,0.526067,2.89119,9.37958,-0.032333,-0.044224,0.011758,17.9510,-4.56875,-41.048800,8
926025,3630.09,5,1.291976,0.870418,2.581465,0.630675,2.81489,9.34943,-0.004483,-0.001148,0.039415,17.9572,-4.94146,-41.177700,8
926026,3630.10,5,1.280116,0.875141,2.554826,0.720712,2.70858,9.40998,-0.022839,0.045628,0.049042,17.7079,-4.80414,-41.309300,8


In [40]:
pamap2[['x_maf', 'y_maf', 'z_maf']] = pamap2.groupby('subject_id')[['hand_acc_16g_x', 'hand_acc_16g_y', 'hand_acc_16g_z']].rolling(window=PAMAP2_HZ).mean().reset_index(drop=True)
pamap2[['x_ema', 'y_ema', 'z_ema']] = pamap2.groupby('subject_id')[['hand_acc_16g_x', 'hand_acc_16g_y', 'hand_acc_16g_z']].ewm(span=1.5*PAMAP2_HZ, adjust=False).mean().reset_index(drop=True)

In [41]:
walking_data = pamap2[(pamap2['activity_id'] == 4) & (pamap2['subject_id'] == 5)]
walking_vectors = walking_data[[x_axis, y_axis, z_axis]].to_numpy()

standing_data = pamap2[(pamap2['activity_id'] == 3) & (pamap2['subject_id'] == 5)]
standing_vectors = standing_data[[x_axis, y_axis, z_axis]].to_numpy()

sitting_data = pamap2[(pamap2['activity_id'] == 2) & (pamap2['subject_id'] == 5)]
sitting_vectors = sitting_data[[x_axis, y_axis, z_axis]].to_numpy()

jogging_data = pamap2[(pamap2['activity_id'] == 5) & (pamap2['subject_id'] == 5)]
jogging_vectors = jogging_data[[x_axis, y_axis, z_axis]].to_numpy()

upstairs_data = pamap2[(pamap2['activity_id'] == 12) & (pamap2['subject_id'] == 5)]
upstairs_vectors = upstairs_data[[x_axis, y_axis, z_axis]].to_numpy()

downstairs_data = pamap2[(pamap2['activity_id'] == 13) & (pamap2['subject_id'] == 5)]
downstairs_vectors = downstairs_data[[x_axis, y_axis, z_axis]].to_numpy()

print(len(walking_vectors)), print(len(standing_vectors)), print(len(sitting_vectors)), print(len(jogging_vectors)), print(len(upstairs_vectors)), print(len(downstairs_vectors))

30909
22125
26826
24586
14275
12723


(None, None, None, None, None, None)

In [42]:
data_x = {"Standing": standing_data[x_axis].to_numpy(), 
          "Walking": walking_data[x_axis].to_numpy(), 
          "Sitting": sitting_data[x_axis].to_numpy(), 
          "Jogging": jogging_data[x_axis].to_numpy(),
          "Downstairs": downstairs_data[x_axis].to_numpy(),
          "Upstairs": upstairs_data[x_axis].to_numpy()
          }

data_y = {"Standing": standing_data[y_axis].to_numpy(), 
          "Walking": walking_data[y_axis].to_numpy(), 
          "Sitting": sitting_data[y_axis].to_numpy(), 
          "Jogging": jogging_data[y_axis].to_numpy(),
            "Downstairs": downstairs_data[y_axis].to_numpy(),
            "Upstairs": upstairs_data[y_axis].to_numpy()
          }

data_z = {"Standing": standing_data[z_axis].to_numpy(), 
          "Walking": walking_data[z_axis].to_numpy(), 
          "Sitting": sitting_data[z_axis].to_numpy(), 
          "Jogging": jogging_data[z_axis].to_numpy(),
            "Downstairs": downstairs_data[z_axis].to_numpy(),
            "Upstairs": upstairs_data[z_axis].to_numpy()
          }

data_vectors = {"Standing": standing_vectors, 
                "Walking": walking_vectors, 
                "Sitting": sitting_vectors, 
                "Jogging": jogging_vectors,
                "Downstairs": downstairs_vectors,
                "Upstairs": upstairs_vectors
                }

In [43]:
STEP = int(WINDOW_SIZE*PAMAP2_HZ)
for i in range(len(threshold_list)):
    plot_recurrence_by_activity(data_x, DATASET_NAME, "X", threshold_list[i], step=STEP, first=PAMAP2_START, filename=f"{RELATIVE_PATH}/{WINDOW_SIZE}/X/recurrence_pamap2_x_{i}-{str(threshold_list[i])}")
    plot_recurrence_by_activity(data_y, DATASET_NAME, "Y", threshold_list[i], step=STEP, first=PAMAP2_START, filename=f"{RELATIVE_PATH}/{WINDOW_SIZE}/Y/recurrence_pamap2_y_{i}-{str(threshold_list[i])}")
    plot_recurrence_by_activity(data_z, DATASET_NAME, "Z", threshold_list[i], step=STEP, first=PAMAP2_START, filename=f"{RELATIVE_PATH}/{WINDOW_SIZE}/Z/recurrence_pamap2_z_{i}-{str(threshold_list[i])}")
    plot_recurrence_by_activity(data_vectors, DATASET_NAME, "Vector", threshold_list[i], step=STEP, first=PAMAP2_START, filename=f"{RELATIVE_PATH}/{WINDOW_SIZE}/Vector/recurrence_pamap2_vector_{i}-{str(threshold_list[i])}")

# Clemson

In [44]:
DATASET_NAME = 'CLEMSON'
RELATIVE_PATH = "C:/Users/tulio/Downloads/Plots/CLEMSON"
CLEMSON_HZ = 15
CLEMSON_START = int(WINDOW_SIZE * CLEMSON_HZ * 3)

In [45]:
# Function to normalize columns
def normalize_group(group):
    group[['acc_x', 'acc_y', 'acc_z']] = (group[['acc_x', 'acc_y', 'acc_z']] - group[['acc_x', 'acc_y', 'acc_z']].mean()) / group[['acc_x', 'acc_y', 'acc_z']].std()
    return group

clemson = pd.read_csv('../../datasets/combined_clemson_cafeteria_data.csv', sep=',')
clemson = clemson.groupby('user_id').apply(normalize_group)
clemson = clemson.reset_index(drop=True)
clemson

C:\Users\tulio\AppData\Local\Temp\ipykernel_2484\2720396042.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  clemson = clemson.groupby('user_id').apply(normalize_group)


,acc_x,acc_y,acc_z,gyr_x,gyr_y,gyr_z,grams,user_id,course,index,activity
0,-3.085055,0.320500,1.546432,65.232882,23.322505,46.738401,828.7,p005,c1,0,other
1,-1.352688,0.644118,1.694550,49.632882,-5.077495,23.938401,828.7,p005,c1,1,other
2,-2.061384,0.666283,1.550435,38.432882,-7.477495,17.938401,828.7,p005,c1,2,other
3,-1.875773,0.555455,1.550435,38.832882,-9.877495,18.738401,828.7,p005,c1,3,other
4,-1.650790,0.608653,1.538426,37.632882,-11.477495,19.138401,828.9,p005,c1,4,other
...,...,...,...,...,...,...,...,...,...,...,...
6367686,1.002465,1.319253,-0.738027,-5.460823,-30.588564,4.409891,826.2,p413,c1,17222,other
6367687,1.292542,1.319253,-0.794112,-5.060823,-19.388564,4.809891,826.3,p413,c1,17223,other
6367688,1.135212,1.482703,-0.797228,-12.660823,-12.988564,0.809891,826.3,p413,c1,17224,other
6367689,1.194211,1.610620,-0.903167,-10.260823,-14.188564,1.209891,826.3,p413,c1,17225,other


In [46]:
clemson[['x_maf', 'y_maf', 'z_maf']] = clemson.groupby('user_id')[['acc_x', 'acc_y', 'acc_z']].rolling(window=CLEMSON_HZ).mean().reset_index(drop=True)
clemson[['x_ema', 'y_ema', 'z_ema']] = clemson.groupby('user_id')[['acc_x', 'acc_y', 'acc_z']].ewm(span=1.5*CLEMSON_HZ, adjust=False).mean().reset_index(drop=True)

In [47]:
clemson['activity'].unique()

array(['other', 'utensiling', 'bite', 'rest', 'drink', nan], dtype=object)

In [48]:
bite_data = clemson[(clemson['activity'] == 'bite') & (clemson['user_id'] == 'p006')]
bite_vectors = bite_data[[x_axis, y_axis, z_axis]].to_numpy()

rest_data = clemson[(clemson['activity'] == 'rest') & (clemson['user_id'] == 'p006')]
rest_vectors = rest_data[[x_axis, y_axis, z_axis]].to_numpy()

drink_data = clemson[(clemson['activity'] == 'drink') & (clemson['user_id'] == 'p006')]
drink_vectors = drink_data[[x_axis, y_axis, z_axis]].to_numpy()

other_data = clemson[(clemson['activity'] == 'other') & (clemson['user_id'] == 'p006')]
other_vectors = other_data[[x_axis, y_axis, z_axis]].to_numpy()

print(len(bite_vectors)), print(len(rest_vectors)), print(len(drink_vectors)), print(len(other_vectors))

2536
5710
1343
9571


(None, None, None, None)

In [49]:
data_x = {"Bite": bite_data[x_axis].to_numpy(), 
          "Rest": rest_data[x_axis].to_numpy(), 
          "Drink": drink_data[x_axis].to_numpy(), 
          "Other": other_data[x_axis].to_numpy(),
          }

data_y = {"Bite": bite_data[y_axis].to_numpy(), 
          "Rest": rest_data[y_axis].to_numpy(), 
          "Drink": drink_data[y_axis].to_numpy(), 
          "Other": other_data[y_axis].to_numpy(),
          }

data_z = {"Bite": bite_data[z_axis].to_numpy(), 
          "Rest": rest_data[z_axis].to_numpy(), 
          "Drink": drink_data[z_axis].to_numpy(), 
          "Other": other_data[z_axis].to_numpy(),
          }

data_vectors = {"Bite": bite_vectors, 
                "Rest": rest_vectors, 
                "Drink": drink_vectors, 
                "Other": other_vectors,
                }

In [50]:
STEP = int(WINDOW_SIZE*CLEMSON_HZ)
for i in range(len(threshold_list)):
    plot_recurrence_by_activity(data_x, DATASET_NAME, "X", threshold_list[i], step=STEP, first=CLEMSON_START, filename=f"{RELATIVE_PATH}/{WINDOW_SIZE}/X/recurrence_clemson_x_{i}-{str(threshold_list[i])}")
    plot_recurrence_by_activity(data_y, DATASET_NAME, "Y", threshold_list[i], step=STEP, first=CLEMSON_START, filename=f"{RELATIVE_PATH}/{WINDOW_SIZE}/Y/recurrence_clemson_y_{i}-{str(threshold_list[i])}")
    plot_recurrence_by_activity(data_z, DATASET_NAME, "Z", threshold_list[i], step=STEP, first=CLEMSON_START, filename=f"{RELATIVE_PATH}/{WINDOW_SIZE}/Z/recurrence_clemson_z_{i}-{str(threshold_list[i])}")
    plot_recurrence_by_activity(data_vectors, DATASET_NAME, "Vector", threshold_list[i], step=STEP, first=CLEMSON_START, filename=f"{RELATIVE_PATH}/{WINDOW_SIZE}/Vector/recurrence_clemson_vector_{i}-{str(threshold_list[i])}")